# ML-Based Scaling Laws: Supervised Metrics as MI Alternatives

Train simple supervised models on embeddings and report scaling of metrics with dataset size and noise quality.

- **shendure** (`author_day`): KNN + Logistic Regression classification -> accuracy, macro F1
- **PBMC** (`protein_counts`): Ridge regression -> mean R², MSE, MAE over 217 proteins
- **merfish** (`ng_idx`): Ridge regression -> cell embedding predicts spatial neighbor's embedding
- **larry** (`clone`): Ridge regression -> early cell embedding predicts late clone mate's embedding

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import anndata as ad
from pathlib import Path
from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

DATA_ROOT = Path('/home/igor/noise_scaling/data')
SEED = 42
N_WORKERS = 16  # keep moderate — large CSV loads are memory-heavy

ALGOS = ['Geneformer', 'PCA', 'RandomProjection', 'SCVI', 'State']
ALGO_ORDER = ['State', 'Geneformer', 'SCVI', 'PCA', 'RandomProjection']

EXPECTED = {
    'PBMC': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.0012346, 0.0025982, 0.0054682, 0.0115083, 0.02422, 0.050973, 0.1072766, 0.225772, 0.4751547, 1.0],
        'signal': 'protein_counts',
        'task': 'regression',
    },
    'larry': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.003876, 0.0071835, 0.0133136, 0.0246748, 0.0457311, 0.0847557, 0.1570821, 0.2911284, 0.5395631, 1.0],
        'signal': 'clone',
        'task': 'regression',
    },
    'merfish': {
        'sizes': [100, 203, 414, 843, 1716, 3494, 7113, 14480, 29475, 60000],
        'qualities': [0.027248, 0.0406617, 0.0606789, 0.0905502, 0.1351267, 0.2016475, 0.3009156, 0.4490518, 0.6701133, 1.0],
        'signal': 'ng_idx',
        'task': 'regression',
    },
    'shendure': {
        'sizes': [100, 359, 1291, 4641, 16681, 59948, 215443, 774263, 2782559, 10000000],
        'qualities': [0.004, 0.0073875, 0.0136438, 0.0251984, 0.0465384, 0.0859506, 0.1587401, 0.2931733, 0.5414548, 1.0],
        'signal': 'author_day',
        'task': 'classification',
    },
}

In [ ]:
# ---------------------------------------------------------------------------
# Path helpers
# ---------------------------------------------------------------------------
def embeddings_path(dataset, size, quality, algorithm):
    return DATA_ROOT / dataset / str(size) / str(quality) / 'results' / algorithm / 'model' / 'embeddings.csv'

def signal_path(dataset, quality, signal, algorithm):
    q = str(quality)
    if algorithm == 'Geneformer':
        return DATA_ROOT / dataset / 'test' / q / 'signals' / f'Y_{signal}_{q}_geneformer.csv'
    return DATA_ROOT / dataset / 'test' / q / 'signals' / f'Y_{signal}_{q}.csv'

# ---------------------------------------------------------------------------
# Dataset-specific evaluation functions
# ---------------------------------------------------------------------------
def evaluate_shendure(emb_path, sig_path):
    """Classification of author_day from embeddings."""
    X = pd.read_csv(emb_path).values.astype(np.float64)
    y_raw = pd.read_csv(sig_path).iloc[:, 0].values
    y = LabelEncoder().fit_transform(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    res = {
        'knn_accuracy': accuracy_score(y_test, y_pred),
        'knn_macro_f1': f1_score(y_test, y_pred, average='macro'),
    }

    lr = LogisticRegression(max_iter=2000, multi_class='multinomial', solver='lbfgs')
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    res['logreg_accuracy'] = accuracy_score(y_test, y_pred)
    res['logreg_macro_f1'] = f1_score(y_test, y_pred, average='macro')
    return res


def evaluate_pbmc(emb_path, sig_path):
    """Ridge regression predicting 217-dim protein counts."""
    X = pd.read_csv(emb_path).values.astype(np.float64)
    Y = pd.read_csv(sig_path).values.astype(np.float64)

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=SEED)
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, Y_train)
    Y_pred = ridge.predict(X_test)

    r2s = [r2_score(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    mses = [mean_squared_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    maes = [mean_absolute_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    return {'ridge_mean_r2': np.mean(r2s), 'ridge_mean_mse': np.mean(mses), 'ridge_mean_mae': np.mean(maes)}


def evaluate_merfish(emb_path, sig_path, quality):
    """Ridge regression: cell embedding -> spatial neighbor's embedding."""
    embeddings_df = pd.read_csv(emb_path)
    signal_df = pd.read_csv(sig_path, dtype=str)
    cur_idx, ng_idx = signal_df.iloc[:, 0].values, signal_df.iloc[:, 1].values

    assert len(cur_idx) == len(embeddings_df)
    assert set(ng_idx).issubset(set(cur_idx))

    embeddings_df.index = cur_idx
    Y = embeddings_df.loc[ng_idx].values

    preprocessed_path = DATA_ROOT / 'merfish' / 'test' / str(quality) / 'preprocessed' / 'preprocessed.h5ad'
    adata = ad.read_h5ad(preprocessed_path, backed='r')
    test_indices = adata.uns['test_indices'].astype(int)
    X = embeddings_df.values
    test_indices = test_indices[test_indices < X.shape[0]]
    X = X[test_indices].astype(np.float64)
    Y = Y[test_indices].astype(np.float64)
    assert X.shape == Y.shape

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=SEED)
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, Y_train)
    Y_pred = ridge.predict(X_test)

    r2s = [r2_score(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    mses = [mean_squared_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    maes = [mean_absolute_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    return {'ridge_mean_r2': np.mean(r2s), 'ridge_mean_mse': np.mean(mses), 'ridge_mean_mae': np.mean(maes)}


def evaluate_larry(emb_path, sig_path):
    """Ridge regression: early clone cell embedding -> late clone mate embedding."""
    embeddings_df = pd.read_csv(emb_path)
    signal_df = pd.read_csv(sig_path, dtype={0: str, 1: str, 2: float})
    signal_df.columns = ['index', 'clone', 'time']

    early = signal_df[signal_df['time'].isin([2, 4])]
    late = signal_df[signal_df['time'].isin([6])]
    common_clones = (
        set(late['clone'].values)
        .intersection(set(early['clone'].values))
        .intersection(set(signal_df['index'].values))
    )
    if len(common_clones) == 0:
        return {'ridge_mean_r2': np.nan, 'ridge_mean_mse': np.nan, 'ridge_mean_mae': np.nan}

    early_common = early[early['clone'].isin(common_clones)].groupby('clone').sample(n=1, random_state=SEED)
    late_common = late[late['clone'].isin(common_clones)].groupby('clone').sample(n=1, random_state=SEED)

    embeddings_df.index = signal_df['index'].values
    X = embeddings_df.loc[early_common['index'].tolist()].values.astype(np.float64)
    Y = embeddings_df.loc[late_common['index'].tolist()].values.astype(np.float64)
    assert X.shape == Y.shape

    if X.shape[0] < 10:
        return {'ridge_mean_r2': np.nan, 'ridge_mean_mse': np.nan, 'ridge_mean_mae': np.nan}

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=SEED)
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, Y_train)
    Y_pred = ridge.predict(X_test)

    r2s = [r2_score(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    mses = [mean_squared_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    maes = [mean_absolute_error(Y_test[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    return {'ridge_mean_r2': np.mean(r2s), 'ridge_mean_mse': np.mean(mses), 'ridge_mean_mae': np.mean(maes)}

In [ ]:
# ---------------------------------------------------------------------------
# Worker function for parallel execution
# ---------------------------------------------------------------------------
def run_single_eval(args):
    """Evaluate a single (dataset, algorithm, size, quality) combination."""
    dataset, algo, size, quality = args
    cfg = EXPECTED[dataset]
    emb_p = embeddings_path(dataset, size, quality, algo)
    sig_p = signal_path(dataset, quality, cfg['signal'], algo)

    if not emb_p.exists() or not sig_p.exists():
        return None

    try:
        if dataset == 'shendure':
            metrics = evaluate_shendure(emb_p, sig_p)
        elif dataset == 'PBMC':
            metrics = evaluate_pbmc(emb_p, sig_p)
        elif dataset == 'merfish':
            metrics = evaluate_merfish(emb_p, sig_p, quality)
        elif dataset == 'larry':
            metrics = evaluate_larry(emb_p, sig_p)
        else:
            return None
    except Exception as e:
        return {'dataset': dataset, 'algorithm': algo, 'size': size,
                'quality': quality, '_error': str(e)}

    return {
        'dataset': dataset, 'algorithm': algo, 'size': size,
        'quality': quality, 'signal': cfg['signal'], 'task': cfg['task'],
        **metrics,
    }

In [ ]:
# ---------------------------------------------------------------------------
# Build job list and run in parallel
# ---------------------------------------------------------------------------
jobs = []
for ds, cfg in EXPECTED.items():
    for algo, size, quality in product(ALGOS, cfg['sizes'], cfg['qualities']):
        jobs.append((ds, algo, size, quality))

print(f'Total jobs: {len(jobs)}, workers: {N_WORKERS}')

results = []
errors = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(run_single_eval, job): job for job in jobs}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Evaluating'):
        result = future.result()
        if result is None:
            continue
        if '_error' in result:
            errors.append(result)
        else:
            results.append(result)

df_results = pd.DataFrame(results)
print(f'\nCollected {len(df_results)} results, {len(errors)} errors')
if errors:
    print(f'First errors:')
    for e in errors[:5]:
        print(f"  {e['dataset']}/{e['algorithm']}/{e['size']}/{e['quality']}: {e['_error'][:80]}")

In [ ]:
# Completeness table
pivot = df_results.groupby(['dataset', 'algorithm']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=ALGO_ORDER, fill_value=0)
print('Results per dataset x algorithm:')
display(pivot)

## Metric vs Quality (noise scaling)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

METRIC_MAP = {
    'shendure': ('knn_accuracy', 'KNN Accuracy'),
    'PBMC': ('ridge_mean_r2', 'Ridge Mean R²'),
    'merfish': ('ridge_mean_r2', 'Ridge Mean R²'),
    'larry': ('ridge_mean_r2', 'Ridge Mean R²'),
}

DS_ORDER = ['PBMC', 'larry', 'merfish', 'shendure']

def plot_metric_vs_quality(df, metric_map=METRIC_MAP):
    """Plot metric vs quality: rows=dataset, cols=algorithm, lines=sizes."""
    datasets = [d for d in DS_ORDER if d in df['dataset'].unique()]
    algorithms = [a for a in ALGO_ORDER if a in df['algorithm'].unique()]
    n_rows, n_cols = len(datasets), len(algorithms)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows + 0.8),
                             squeeze=False, sharex=False)

    for i in range(n_rows):
        for j in range(1, n_cols):
            axes[i, j].sharey(axes[i, 0])

    for i, ds in enumerate(datasets):
        metric_col, metric_label = metric_map[ds]
        ds_sizes = sorted(df[df['dataset'] == ds]['size'].unique())
        if not ds_sizes:
            continue
        norm = mcolors.LogNorm(vmin=min(ds_sizes), vmax=max(ds_sizes))
        cmap = plt.cm.viridis

        for j, algo in enumerate(algorithms):
            ax = axes[i, j]
            sub = df[(df['dataset'] == ds) & (df['algorithm'] == algo)].dropna(subset=[metric_col])

            if sub.empty:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=10)
            else:
                for sz in sorted(sub['size'].unique()):
                    color = cmap(norm(sz))
                    sz_df = sub[sub['size'] == sz].sort_values('quality')
                    ax.plot(sz_df['quality'], sz_df[metric_col],
                            marker='o', markersize=3, linewidth=1, color=color, alpha=0.85)

            ax.set_xscale('log')
            if i == n_rows - 1:
                ax.set_xlabel('Quality (downsampling ratio)', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'{ds} — {metric_label}', fontsize=10)
            if i == 0:
                ax.set_title(algo, fontsize=12, fontweight='bold')
            ax.tick_params(labelsize=8)
            if j > 0:
                ax.tick_params(labelleft=False)

    # Colorbar
    all_sizes = sorted(df['size'].unique())
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis,
                                norm=mcolors.LogNorm(vmin=min(all_sizes), vmax=max(all_sizes)))
    sm.set_array([])
    cbar_ax = fig.add_axes([0.55, 0.01, 0.35, 0.015])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Number of cells', fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    fig.suptitle('ML Metrics vs Quality', fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()
    return fig

fig_quality = plot_metric_vs_quality(df_results)

## Save results

In [ ]:
out_path = Path('2026-04-15_15-00_ml_scaling_results.csv')
df_results.to_csv(out_path, index=False)
print(f'Saved {len(df_results)} results to {out_path}')
df_results.head()